In [1]:
import os
%pwd

'/home/tuhin/bangla-political-memes-classification/research'

In [2]:
os.chdir("../")

In [3]:
%pwd

'/home/tuhin/bangla-political-memes-classification'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TextExtractionConfig:
    root_dir: Path
    train_csv_path: Path
    train_image_folder: Path
    test_csv_path: Path
    test_image_folder: Path
    extracted_train_csv: Path
    extracted_test_csv: Path

In [5]:
from memeClassifier.constants import *
from memeClassifier.utils.common import read_yaml, create_directories

In [6]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_text_extraction_config(self) -> TextExtractionConfig:
        config = self.config.text_extraction

        create_directories([config.root_dir])

        text_extraction_config = TextExtractionConfig(
            root_dir=Path(config.root_dir),
            train_csv_path=Path(config.train_csv_path),
            train_image_folder=Path(config.train_image_folder),
            test_csv_path=Path(config.test_csv_path),
            test_image_folder=Path(config.test_image_folder),
            extracted_train_csv=Path(config.extracted_train_csv),
            extracted_test_csv=Path(config.extracted_test_csv)
        )

        return text_extraction_config

In [7]:
import easyocr
import pandas as pd
from pathlib import Path
import os
from tqdm import tqdm
from memeClassifier import logger

In [8]:
class TextExtraction:
    def __init__(self, config: TextExtractionConfig):
        self.config = config
        logger.info("Initializing EasyOCR reader for Bengali and English...")
        self.reader = easyocr.Reader(['bn', 'en'], gpu=False)
        logger.info("✓ Reader initialized")
        
    def extract_text_and_save(self, csv_path: Path, image_folder: Path, output_path: Path):
        df = pd.read_csv(csv_path)
        logger.info(f"Loaded {len(df)} images from {csv_path}")
        
        extracted_texts = []
        for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing images"):
            image_name = row['Image_name']
            image_path = os.path.join(image_folder, image_name)
            
            try:
                if os.path.exists(image_path):
                    result = self.reader.readtext(image_path)
                    text = ' '.join([detection[1] for detection in result])
                    extracted_texts.append(text)
                else:
                    logger.warning(f"Image not found: {image_path}")
                    extracted_texts.append("")
            except Exception as e:
                logger.error(f"Error processing {image_path}: {e}")
                extracted_texts.append("")
                
        df['Extracted_Text'] = extracted_texts
        df.to_csv(output_path, index=False)
        
        logger.info(f"✓ Saved results to {output_path}")
        logger.info(f"Statistics for {output_path}:")
        logger.info(f"  Total images processed: {len(df)}")
        logger.info(f"  Images with extracted text: {(df['Extracted_Text'] != '').sum()}")
        logger.info(f"  Images with no text: {(df['Extracted_Text'] == '').sum()}")
        
    def initiate_text_extraction(self):
        logger.info("Extracting text for training data")
        self.extract_text_and_save(
            self.config.train_csv_path,
            self.config.train_image_folder,
            self.config.extracted_train_csv
        )
        
        logger.info("Extracting text for testing data")
        self.extract_text_and_save(
            self.config.test_csv_path,
            self.config.test_image_folder,
            self.config.extracted_test_csv
        )

In [9]:
try:
    config = ConfigurationManager()
    text_extraction_config = config.get_text_extraction_config()
    text_extraction = TextExtraction(config=text_extraction_config)
    text_extraction.initiate_text_extraction()
except Exception as e:
    raise e

[2026-06-26 16:30:20,808: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-06-26 16:30:20,812: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-26 16:30:20,813: INFO: common: Directory created at: artifacts]
[2026-06-26 16:30:20,814: INFO: common: Directory created at: artifacts/text_extraction]
[2026-06-26 16:30:20,815: INFO: 4234168340: Initializing EasyOCR reader for Bengali and English...]
[2026-06-26 16:30:20,815: WARNING: easyocr: Using CPU. Note: This module is much faster with a GPU.]
[2026-06-26 16:30:23,992: INFO: 4234168340: ✓ Reader initialized]
[2026-06-26 16:30:23,992: INFO: 4234168340: Extracting text for training data]
[2026-06-26 16:30:23,996: INFO: 4234168340: Loaded 195 images from artifacts/data_ingestion/meme_classification_dataset/Train/Train.csv]


Processing images:   0%|          | 0/195 [00:00<?, ?it/s]/home/tuhin/bangla-political-memes-classification/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Processing images:   1%|          | 1/195 [00:03<11:10,  3.46s/it]/home/tuhin/bangla-political-memes-classification/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Processing images:   1%|          | 2/195 [00:05<08:47,  2.73s/it]/home/tuhin/bangla-political-memes-classification/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Processin

[2026-06-26 16:35:29,018: WARNING: 4234168340: Image not found: artifacts/data_ingestion/meme_classification_dataset/Train/Image/train0057.jpg]


/home/tuhin/bangla-political-memes-classification/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Processing images:  29%|██▉       | 57/195 [05:10<10:51,  4.72s/it]/home/tuhin/bangla-political-memes-classification/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Processing images:  30%|██▉       | 58/195 [05:17<12:00,  5.26s/it]/home/tuhin/bangla-political-memes-classification/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Processing images:  30%|███       | 59/195 [05:23<11:55,  5.26s/i

[2026-06-26 16:42:04,481: WARNING: 4234168340: Image not found: artifacts/data_ingestion/meme_classification_dataset/Train/Image/train0105.jpg]


...

[2026-06-26 16:51:24,690: WARNING: 4234168340: Image not found: artifacts/data_ingestion/meme_classification_dataset/Train/Image/train0197.jpg]


/home/tuhin/bangla-political-memes-classification/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Processing images:  99%|█████████▉| 193/195 [21:03<00:05,  2.93s/it]/home/tuhin/bangla-political-memes-classification/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Processing images:  99%|█████████▉| 194/195 [21:08<00:03,  3.32s/it]/home/tuhin/bangla-political-memes-classification/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Processing images: 100%|██████████| 195/195 [21:12<00:00,  6.53

[2026-06-26 16:51:36,674: INFO: 4234168340: ✓ Saved results to artifacts/text_extraction/meme_train_data_with_text.csv]
[2026-06-26 16:51:36,675: INFO: 4234168340: Statistics for artifacts/text_extraction/meme_train_data_with_text.csv:]
[2026-06-26 16:51:36,676: INFO: 4234168340:   Total images processed: 195]
[2026-06-26 16:51:36,677: INFO: 4234168340:   Images with extracted text: 191]
[2026-06-26 16:51:36,678: INFO: 4234168340:   Images with no text: 4]
[2026-06-26 16:51:36,679: INFO: 4234168340: Extracting text for testing data]
[2026-06-26 16:51:36,683: INFO: 4234168340: Loaded 100 images from artifacts/data_ingestion/meme_classification_dataset/Test/Test.csv]



Processing images:   0%|          | 0/100 [00:00<?, ?it/s]/home/tuhin/bangla-political-memes-classification/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Processing images:   1%|          | 1/100 [00:08<14:00,  8.49s/it]/home/tuhin/bangla-political-memes-classification/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Processing images:   2%|▏         | 2/100 [00:17<14:42,  9.00s/it]/home/tuhin/bangla-political-memes-classification/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Processi

[2026-06-26 17:11:42,762: INFO: 4234168340: ✓ Saved results to artifacts/text_extraction/meme_test_data_with_text.csv]
[2026-06-26 17:11:42,763: INFO: 4234168340: Statistics for artifacts/text_extraction/meme_test_data_with_text.csv:]
[2026-06-26 17:11:42,763: INFO: 4234168340:   Total images processed: 100]
[2026-06-26 17:11:42,764: INFO: 4234168340:   Images with extracted text: 100]
[2026-06-26 17:11:42,765: INFO: 4234168340:   Images with no text: 0]
